# IALS (Implicit Alternating Least Squares )

В этом ноутбуке я построю IALS модель, протестирую её и сохраню 

## Импорт библиотек 

In [19]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
import scipy 
import os
os.environ['OPENBLAS_NUM_THREADS'] = '1'
import implicit 
from implicit.nearest_neighbours import bm25_weight
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from implicit.evaluation import train_test_split as implicit_split 
from implicit.evaluation import precision_at_k, mean_average_precision_at_k, ranking_metrics_at_k
import implicit.gpu
import optuna
import pickle
import optuna.visualization as vis
import optuna.visualization.matplotlib as vis_matplotlib
import sqlite3
import plotly
import time


In [2]:
plt.style.use('default')
sns.set_palette("husl")
%matplotlib inline

## Загрузка данных

#### Выгружаем матрицу

In [3]:
results_path = Path("../../../results/matrices")
train_sparse = scipy.sparse.load_npz(results_path/'artnail_user_item_sparse_train.npz')
val_sparse = scipy.sparse.load_npz(results_path/'artnail_user_item_sparse_val.npz')
test_sparse = scipy.sparse.load_npz(results_path/'artnail_user_item_sparse_test.npz')

#### Выгружаем очищенные таблицы

In [4]:
clean_path = Path("../../../Tables/CleanTable")
users_clean = pd.read_csv(clean_path / 'users_clean.csv')
items_dim = pd.read_csv(clean_path / 'items_dim.csv')

In [5]:
users_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 2711 entries, 0 to 2710
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_user          2711 non-null   int64  
 1   Имя              2711 non-null   str    
 2   Телефон          2711 non-null   str    
 3   Категории        2711 non-null   str    
 4   Оплачено в руб   2711 non-null   float64
 5   Последний визит  2711 non-null   str    
dtypes: float64(1), int64(1), str(4)
memory usage: 127.2 KB


In [6]:
items_dim.info()

<class 'pandas.DataFrame'>
RangeIndex: 235 entries, 0 to 234
Data columns (total 3 columns):
 #   Column                                 Non-Null Count  Dtype
---  ------                                 --------------  -----
 0   id_item                                235 non-null    int64
 1   Название услуги или товара             235 non-null    str  
 2   Категория товара или услуги в продаже  235 non-null    str  
dtypes: int64(1), str(2)
memory usage: 5.6 KB


In [7]:
print(f'Размеры train матрицы: {train_sparse.shape}',
       f'Размеры val матрицы: {val_sparse.shape}',
         f'Размеры test матрицы: {test_sparse.shape}', sep='\n')

Размеры train матрицы: (1265, 161)
Размеры val матрицы: (1265, 161)
Размеры test матрицы: (1265, 161)


## IALS model 

#### Построение базовой модели 

In [8]:
IALS_model = implicit.als.AlternatingLeastSquares(
    factors=4,           
    regularization=100,   
    iterations=20,        
    random_state=42,      
    use_gpu=False,        
    num_threads=4         
)

c:\obuchenie\ArtNail\.venv\Lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: OpenBLAS is configured to use 16 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


Обучение 

In [9]:
#ALPHA = 40
#IALS_model.fit(train_sparse * ALPHA)

In [11]:
ALPHA = 40
train_weighted = bm25_weight(train_sparse, K1=1.5, B=0.75) * ALPHA
IALS_model.fit(train_weighted)

print(f"User embeddings: {IALS_model.user_factors.shape}")
print(f"Item embeddings: {IALS_model.item_factors.shape}")

c:\obuchenie\ArtNail\.venv\Lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.0 seconds
  warnings.warn(


  0%|          | 0/20 [00:00<?, ?it/s]

User embeddings: (1265, 4)
Item embeddings: (161, 4)


#### Оценка качества базовой модели 

IALS будет выполнять функции подбора кандидатов, а функцию ранжирования будет выполнять `CutBoost` модель, поэтому в оценке `IALS` главной метрикой буду считать Recall@K, где K будет большим.

Цель не отобрать конкретные 10 лучших вариантов, а отобрать 50 кандидатов, главное чтобы среди них были хорошие кандидаты и CutBoost увидел их

In [12]:
def hit_rate_at_k(model, train_matrix, test_matrix, K):
    hits = 0
    users_with_test = test_matrix.nonzero()[0]
    
    for u_idx in users_with_test:
        # Рекомендуем, исключая то, что уже было в трейне
        rec, _ = model.recommend(u_idx, train_matrix[u_idx], N=K, filter_already_liked_items=True)
        
        actual = set(test_matrix[u_idx].indices)
        if set(rec).intersection(actual):
            hits += 1
            
    return hits / len(users_with_test)


In [13]:
hit_rate_at_k(IALS_model, train_sparse, test_sparse, K=10)

0.6375

In [15]:
#empty_train = sp.csr_matrix(train_sparse.shape)
metrics = ranking_metrics_at_k(IALS_model, test_sparse, test_sparse, K=10)
metrics

  0%|          | 0/59 [00:00<?, ?it/s]

{'precision': 0.0, 'map': 0.0, 'ndcg': 0.0, 'auc': 0.4686796190509233}

In [16]:
print(f"Записей в train_sparse: {train_sparse.nnz}")
print(f"Записей в test_sparse: {test_sparse.nnz}")
print(f"Записей в val_sparse: {val_sparse.nnz}")

# Проверка: не пустые ли они?
if test_sparse.nnz != 80:
    print("ВНИМАНИЕ: test_sparse содержит не 80 записей! Проверьте сборку матрицы.")

Записей в train_sparse: 3836
Записей в test_sparse: 80
Записей в val_sparse: 494


In [17]:
# Превращаем матрицы в наборы координат (row, col)
train_coords = set(zip(*train_sparse.nonzero()))
test_coords = set(zip(*test_sparse.nonzero()))

intersection = train_coords.intersection(test_coords)

print(f"Пересечение: {len(intersection)} общих пар (user, item)")
print(f"Процент утечки: {len(intersection) / len(test_coords):.2%}")

if len(intersection) > 0:
    print("ОШИБКА: Тестовые данные содержатся в тренировочных. Метрики завышены из-за этого.")

Пересечение: 0 общих пар (user, item)
Процент утечки: 0.00%


__Из 50 кандидатов 42% релевантны__. Это средний показатель для CutBoost. 

Также  `AUC` = 0.6248 показывает, что модель уверенно отличает положительные и негативные оценки.

Из остальных метрик можно понять, что модель плохо ранжирует отобранных кандидатов, но для нашей ситуации это не страшно, т. к. ранжировать кандидатов будет CutBoost.

#### Random Search с  помощью Optuna

In [ ]:
def objective(trial):
    '''Optuna objective: максимизируем HitRate@10'''
    weight_type = trial.suggest_categorical('weight_type', ['alpha', 'bm25'])
    factors = trial.suggest_int('factors', 4, 24, step=4)
    reg = trial.suggest_float('regularization', 0.1, 100, log=True)

    if weight_type == 'alpha':
        alpha_val = trial.suggest_float('alpha_val', 10, 100, log=True)
        train_weighted = train_sparse * alpha_val
    else:
        k1 = trial.suggest_float('k1', 1.0, 2.0)
        b = trial.suggest_float('b', 0.5, 1.0)
        train_weighted = bm25_weight(train_sparse, K1=k1, B=b)

    model = implicit.als.AlternatingLeastSquares(
        factors=factors,
        regularization=reg,
        iterations=20, 
        random_state=42,
        num_threads=4
    )
    
    model.fit(train_weighted, show_progress=False)
    
    hr10 = hit_rate_at_k(model, train_sparse, val_sparse, K=10)
    
    print(f"Trial {trial.number}: HR@10={hr10:.4f} | weight_type={weight_type}, factors={factors}, reg={reg:.4e}")
    return hr10

In [23]:

study = optuna.create_study(
    study_name='IALS Optuna Optimization',
    direction='maximize',
    storage='sqlite:///ex.db',
    load_if_exists=True
)

[I 2026-04-16 09:53:51,365] Using an existing study with name 'IALS Optuna Optimization' instead of creating a new one.


In [24]:

""" 
def objective(trial):
    
    # Предлагаем параметры
    params = {
    # Размер эмбеддингов: от компактных до довольно крупных
    # (чем больше factors, тем качественнее, но медленнее и больше памяти)
    'factors': trial.suggest_int('factors', 32, 160, step=16),  
    # 32, 48, 64, 80, 96, 112, 128, 144, 160

    # Регуляризация: от очень слабой до довольно сильной
    # (я бы уже использовал log-шкалу — так лучше исследуются края)
    'regularization': trial.suggest_float('regularization', 0.01, 0.5),

    # Число итераций: чуть меньше базового до заметно больше
    # (если данные не очень большие, можно позволить до 80)
    'iterations': trial.suggest_int('iterations', 30, 80),

    # Дополнительные параметры ALS:
    # weight для нормализации регуляризации по числу факторов
    'dtype': np.float32,  # чтобы каждый trial был одинаков по типу
    'use_gpu': False,
    'num_threads': 4,
    'random_state': 42,
    'calculate_training_loss': False
}

    # Модель
    model = implicit.als.AlternatingLeastSquares(**params)
    
    # Обучение 
    CONFIDENCE_SCALE = 15
    model.fit(train_sparse*CONFIDENCE_SCALE, show_progress=False)

    # Метрики
    metrics = ranking_metrics_at_k(
        model, 
        train_user_items=train_sparse, 
        test_user_items=val_sparse, 
        K=50,
        show_progress=False
    )

    recall50 = metrics['precision']

     
    # Callback для pruning (Optuna может прервать плохой trial)
    trial.report(recall50, step=1)
    if trial.should_prune():
        raise optuna.TrialPruned()
    
    
    print(f"✅ Trial {trial.number}: Recall@50={recall50:.4f} | {params}")
    return recall50
"""

' \ndef objective(trial):\n\n    # Предлагаем параметры\n    params = {\n    # Размер эмбеддингов: от компактных до довольно крупных\n    # (чем больше factors, тем качественнее, но медленнее и больше памяти)\n    \'factors\': trial.suggest_int(\'factors\', 32, 160, step=16),  \n    # 32, 48, 64, 80, 96, 112, 128, 144, 160\n\n    # Регуляризация: от очень слабой до довольно сильной\n    # (я бы уже использовал log-шкалу — так лучше исследуются края)\n    \'regularization\': trial.suggest_float(\'regularization\', 0.01, 0.5),\n\n    # Число итераций: чуть меньше базового до заметно больше\n    # (если данные не очень большие, можно позволить до 80)\n    \'iterations\': trial.suggest_int(\'iterations\', 30, 80),\n\n    # Дополнительные параметры ALS:\n    # weight для нормализации регуляризации по числу факторов\n    \'dtype\': np.float32,  # чтобы каждый trial был одинаков по типу\n    \'use_gpu\': False,\n    \'num_threads\': 4,\n    \'random_state\': 42,\n    \'calculate_training_loss

In [25]:
""" 
study = optuna.create_study(
    study_name='IALS Optuna Optimization',
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(),
    storage='sqlite:///ex.db',
    load_if_exists=True
)"""

" \nstudy = optuna.create_study(\n    study_name='IALS Optuna Optimization',\n    direction='maximize',\n    pruner=optuna.pruners.MedianPruner(),\n    storage='sqlite:///ex.db',\n    load_if_exists=True\n)"

In [ ]:
study.optimize(objective, n_trials=50)
print(f"Лучший HitRate@10: {study.best_value}")


[I 2026-03-12 18:21:08,826] Trial 0 finished with value: 0.46601941747572817 and parameters: {'factors': 144, 'regularization': 0.056453401122281165, 'iterations': 33}. Best is trial 0 with value: 0.46601941747572817.


✅ Trial 0: Recall@50=0.4660 | {'factors': 144, 'regularization': 0.056453401122281165, 'iterations': 33, 'dtype': <class 'numpy.float32'>, 'use_gpu': False, 'num_threads': 4, 'random_state': 42, 'calculate_training_loss': False}


[I 2026-03-12 18:21:10,004] Trial 1 finished with value: 0.4077669902912621 and parameters: {'factors': 64, 'regularization': 0.17130586989807609, 'iterations': 31}. Best is trial 0 with value: 0.46601941747572817.


✅ Trial 1: Recall@50=0.4078 | {'factors': 64, 'regularization': 0.17130586989807609, 'iterations': 31, 'dtype': <class 'numpy.float32'>, 'use_gpu': False, 'num_threads': 4, 'random_state': 42, 'calculate_training_loss': False}


[I 2026-03-12 18:21:12,025] Trial 2 finished with value: 0.49514563106796117 and parameters: {'factors': 48, 'regularization': 0.014516395616695727, 'iterations': 60}. Best is trial 2 with value: 0.49514563106796117.


...

✅ Trial 45: Recall@50=0.5922 | {'factors': 112, 'regularization': 0.422630826120688, 'iterations': 51, 'dtype': <class 'numpy.float32'>, 'use_gpu': False, 'num_threads': 4, 'random_state': 42, 'calculate_training_loss': False}


[I 2026-03-12 18:23:53,036] Trial 46 pruned. 
[I 2026-03-12 18:23:56,436] Trial 47 pruned. 
[I 2026-03-12 18:23:59,241] Trial 48 finished with value: 0.5922330097087378 and parameters: {'factors': 96, 'regularization': 0.48302278151794326, 'iterations': 36}. Best is trial 10 with value: 0.6019417475728155.


✅ Trial 48: Recall@50=0.5922 | {'factors': 96, 'regularization': 0.48302278151794326, 'iterations': 36, 'dtype': <class 'numpy.float32'>, 'use_gpu': False, 'num_threads': 4, 'random_state': 42, 'calculate_training_loss': False}


[I 2026-03-12 18:24:03,635] Trial 49 pruned. 


Лучший Recall@50: 0.6019417475728155


In [29]:
#best_params = study.best_params
best_params = {'factors': 96, 'regularization': 0.468162720686304, 'iterations': 79}
f'best_params = {best_params}'

"best_params = {'factors': 96, 'regularization': 0.468162720686304, 'iterations': 79}"

#### Визцализация optuna

In [30]:
vis.plot_optimization_history(study).show()

In [31]:
vis.plot_parallel_coordinate(study).update_layout(title='Успешные комбинации факторов').show()

In [32]:
vis.plot_contour(study, params=['factors', 'regularization']).show()


#### Получение оптимальной модели

In [33]:
best_IALS_model = implicit.als.AlternatingLeastSquares(**best_params, 
                                                    dtype = np.float32,
                                                    use_gpu = False,
                                                    num_threads = 4,
                                                    random_state = 42,
                                                    calculate_training_loss= False
                                                    )

In [34]:
CONFIDENCE_SCALE = 15
best_IALS_model.fit(train_sparse*CONFIDENCE_SCALE)

  0%|          | 0/79 [00:00<?, ?it/s]

Новые метрики

In [35]:
best_metrics = ranking_metrics_at_k(best_IALS_model, train_sparse, val_sparse, K=50)
best_metrics

  0%|          | 0/94 [00:00<?, ?it/s]

{'precision': 0.6116504854368932,
 'map': 0.06785254853016182,
 'ndcg': 0.17672106146494698,
 'auc': 0.7236291904700105}

Сравнение новых и старых метрик 

In [36]:
pd.DataFrame({
    'Metric':['recall', 'map', 'nscg', 'auc'],
    'DefaultModel': [metrics['precision'], metrics['map'], metrics['ndcg'], metrics['auc']],
    'BestModel': [best_metrics['precision'], best_metrics['map'], best_metrics['ndcg'], best_metrics['auc']]
})

,Metric,DefaultModel,BestModel
0,recall,0.427184,0.611650
1,map,0.029478,0.067853
2,nscg,0.106710,0.176721
3,auc,0.624882,0.723629


Видим: 
1) __Вырос recall@50__ ( он же precision в этйо версии) с 0,42 до 0,61
2) Остальные параметры тоже высорли (хоть модель всё ещё плохо ранжирует кандидатов, нам это не критично)
3) __В среднем прирост около 42%__

Что это значит прикладно:

__Теперь из 50 кандидатов 61% реливантны.__

#### Проверка на переобучение 

Проверка происходит на метрике `AUC` т.к. она не зависит от фильтрации в implicit (по умолчаниб стоить filter_already_liked_items=True и она просто отбрасывает train), в отличии от precision/recall, которые на train равны 0.

In [37]:
train_metrics = ranking_metrics_at_k(best_IALS_model, train_sparse, train_sparse, K=50)
print(f"Train AUC:  {train_metrics['auc']:.4f}")
print(f"Val   AUC:  {best_metrics['auc']:.4f}")
print(f"GAP:       {train_metrics['auc'] - metrics['auc']:.4f}")

  0%|          | 0/681 [00:00<?, ?it/s]

Train AUC:  0.4026
Val   AUC:  0.7236
GAP:       -0.2223


Видим, что __модель не переобучена__:

1) `Train AUC` = 0.4 — модель не заучила train
2) `Val AUC` = 0.72 — модель хорошо работает на новых данных
3) Адекватный `GAP` = -0,22 (метрики на тренировочных меньше это хороший знак)

## Сохранение модели 

In [38]:
project_root = Path.cwd().parent.parent.parent
models_dir = project_root/"results"/"models"

model_path = models_dir / "IALS_model.pkl"
models_dir.mkdir(parents=True, exist_ok=True)


In [39]:
with open(model_path, "wb") as f:
    pickle.dump(best_IALS_model, f)